In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# Overview
- MNIST dataset
  + Goal: recognizing handwritten digits
  + Input: a 28x28 image
  + Output: class label (0, 1,..., 9)
- Defining an CNN+MLP for MNIST
- Training and evaluating an CNN+MLP

# Import libraries

In [ ]:
from torchvision import datasets, transforms

transform = transforms.ToTensor()
train_set = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST('./data', train=False, download=True, transform=transform)
x_train = train_set.data.float().unsqueeze(1).div(255).to(device)
y_train = train_set.targets.long().to(device)
x_test = test_set.data.float().unsqueeze(1).div(255).to(device)
y_test = test_set.targets.long().to(device)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

In [ ]:
num_classes = 10

# (1) Data

In [ ]:
class MNISTCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.2))
        self.fc1 = nn.Linear(256 * 3 * 3, 512)
        self.dropout1 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(512, 512)
        self.dropout2 = nn.Dropout(0.2)
        self.fc3 = nn.Linear(512, num_classes)

    def forward(self, x, return_features=False):
        x = self.features(x).flatten(1)
        x = self.dropout1(torch.relu(self.fc1(x)))
        features = self.dropout2(torch.relu(self.fc2(x)))
        logits = self.fc3(features)
        return (logits, features) if return_features else logits

model = MNISTCNN().to(device)
print(model)

In [ ]:
lr = 0.01
batch_size = 128
epochs = 30

# (2) Declare model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.0)
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=batch_size)

def evaluate(loader):
    model.eval(); total_loss = 0.0; correct = 0; count = 0
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb); total_loss += criterion(logits, yb).item() * yb.size(0)
            correct += (logits.argmax(1) == yb).sum().item(); count += yb.size(0)
    return total_loss / count, correct / count

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
starting_time = time.time()
for epoch in range(epochs):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad(); loss_value = criterion(model(xb), yb); loss_value.backward(); optimizer.step()
    train_loss, train_acc = evaluate(train_loader)
    val_loss, val_acc = evaluate(test_loader)
    history['loss'].append(train_loss); history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss); history['val_accuracy'].append(val_acc)
    print(f'Epoch {epoch + 1}/{epochs}: loss={train_loss:.4f}, accuracy={train_acc:.4f}, val_accuracy={val_acc:.4f}')
print('> training time is %.4f minutes' % ((time.time() - starting_time) / 60))

In [ ]:
test_loss, test_accuracy = evaluate(test_loader)
print('Test loss:', test_loss)
print('Test accuracy:', test_accuracy)

random_test_index = np.random.randint(len(test_set))
test_img = x_test[random_test_index:random_test_index + 1]
test_label = y_test[random_test_index].item()
model.eval()
with torch.no_grad():
    pred_label = model(test_img).argmax(1).item()
plt.imshow(test_img[0, 0].cpu(), cmap='gray')
plt.title(f'true label = {test_label}, predicted label = {pred_label}')
plt.show()

# (3) Train defined model
- Note that the training history including loss and accuracy will be save in 'history' variable
- In case your system runs out of memory (OOM), try to decrease batch size

In [ ]:
# Intermediate features from the convolutional network
model.eval()
with torch.no_grad():
    logits, y_feat = model(x_train, return_features=True)
print('feature shape:', y_feat.shape)

from sklearn.decomposition import PCA
pca_fc2 = PCA(n_components=2)
fc2_pca = pca_fc2.fit_transform(y_feat.cpu().numpy())
plt.figure(figsize=(8, 8))
plt.scatter(fc2_pca[:, 0], fc2_pca[:, 1], c=y_train.cpu().numpy(), cmap='viridis', s=2)
plt.title('Visualization of CNN features using PCA')
plt.show()

### Training time
- Training time with GTX 1080 is about 4.2231 minutes for 30 epochs
- Training time with Core i7 is around 105.5 minutes for 30 epochs

# (4) Evaluate trained model